# ZAP scan strategy comparison: passive baseline, active full scan, and narrowed-context active scans

> last_verified: 2026-10-02 · tool_version: n/a (no ZAP release was researched this cycle; the notebook makes no externally-verifiable claims)

## Purpose

A team running the ZAP automation framework does not choose between "scan" and "no scan" — it chooses between *strategies*, and each strategy trades coverage against wall-clock time and against how much state the target is allowed to have changed. The three strategies this notebook compares are:

1. **Passive baseline** — crawl with the spider, raise alerts from passive rules only, write a report.
2. **Active full scan** — spider, then a browser-driven spider, then the active scanner, then a report.
3. **Narrowed-context active scan** — skip discovery entirely and seed the scan context from a supplied endpoint list, then run the active scanner against that list.

The notebook does three things: it models the strategies as comparable attribute sets, it scores them on coverage / speed / blast radius with weights a team can retune, and it reads the plan files that already ship in this kit's ZAP scaffold so the job-graph differences between the strategies are read off real files rather than remembered. It closes with a delta comparison over two report files, which is the shape a promotion gate actually needs.

The cells run with the Python standard library only. ZAP itself is not required to execute this notebook — the comparison logic is the deliverable, and ZAP is only needed to produce the report files the last comparison step consumes.


## When to Use

- Choosing the gate for a new pipeline stage: whether a deploy check should be a passive baseline or an active scan.
- Promoting an existing baseline gate to an active gate ahead of a release, and estimating what changes in runtime when it is promoted.
- Deciding between crawling a target and seeding it with a known endpoint list, when the endpoint inventory already exists and discovery is the slow part.
- Reviewing why two runs of "the same" scan produced different alert counts — the job graphs and the report contents are read separately so the difference can be attributed.
- Auditing whether a plan that looks comprehensive is actually ordered so discovery finishes before active scanning starts.

Not the right tool when the question is about a single finding's evidence, or about tuning one policy rule's threshold — those belong to the plan-structure and scan-configuration documents in `zap/docs/`.


## Prerequisites

- Python 3.9 or later to run the notebook cells (standard library only).
- Read access to this kit's ZAP scaffold for the job-graph cell: `zap/templates/zap-dast-integration-scaffold/`, containing `zap-automation-plan.yaml` and `plans/*.yaml`.
- For the findings-delta step, two report files produced by ZAP report jobs. The cell falls back to clearly-labelled fixtures when none are supplied, so the notebook stays runnable without a scan.
- Run the notebook from the kit root or from `zap/`; the path resolver in Step 3 accepts either.


## Step 1: Model the strategies as comparable attributes

Every downstream cell reads from `STRATEGIES`. The crawl description and the time budgets are modelling choices for this comparison, not vendor measurements — they are the knobs a team retunes per target, so keeping them in one dict means Steps 2 and 5 stay unchanged when the budgets move.

In [ ]:
# last_verified: 2026-10-02 · OWASP ZAP n/a
"""Strategy attributes shared by every comparison cell in this notebook.

The values here are modelling choices for this comparison, not vendor-supplied
measurements: crawl shape, per-scan time budget and blast radius are the knobs a
team tunes per target, so they live in one place and the later cells read them
rather than hard-coding their own numbers.
"""

import json
from pathlib import Path

STRATEGIES = {
    "passive-baseline": {
        "crawl": "spider",
        "active_scan": False,
        "state_changing_requests": False,
        "budget_minutes": 5,
        "finds_response_header_issues": True,
        "finds_input_handling_issues": False,
        "acceptable_on_unowned_targets": True,
    },
    "active-full": {
        "crawl": "spider plus a browser-driven spider",
        "active_scan": True,
        "state_changing_requests": True,
        "budget_minutes": 40,
        "finds_response_header_issues": True,
        "finds_input_handling_issues": True,
        "acceptable_on_unowned_targets": False,
    },
    "narrowed-context-active": {
        "crawl": "no crawl; context seeded from a supplied endpoint list",
        "active_scan": True,
        "state_changing_requests": True,
        "budget_minutes": 15,
        "finds_response_header_issues": False,
        "finds_input_handling_issues": True,
        "acceptable_on_unowned_targets": False,
    },
}

print(json.dumps(STRATEGIES, indent=2))

## Step 2: Score the strategies on coverage, speed and blast radius

Each axis is normalised to 0–1 and combined with the weights in `WEIGHTS`. Blast radius is scored as *safety*: a strategy that is acceptable against an unowned target scores 1.0, and one that sends state-changing requests against such a target scores 0.0. Change `WEIGHTS` to re-rank — a nightly job that must not touch state should raise `blast_radius`; a release gate that cares only about coverage should raise `coverage`.

In [ ]:
WEIGHTS = {"coverage": 0.4, "speed": 0.25, "blast_radius": 0.35}
BUDGET_MINUTES = 60  # normalisation ceiling for the speed axis


def coverage_ratio(strategy):
    covered = int(strategy["finds_response_header_issues"]) + int(
        strategy["finds_input_handling_issues"]
    )
    return covered / 2.0


def speed_ratio(strategy):
    return 1.0 - min(strategy["budget_minutes"] / float(BUDGET_MINUTES), 1.0)


def blast_radius_ratio(strategy):
    """Higher is safer, not smaller."""
    if strategy["acceptable_on_unowned_targets"]:
        return 1.0
    return 0.0 if strategy["state_changing_requests"] else 0.4


def score(strategy, weights=WEIGHTS):
    parts = {
        "coverage": coverage_ratio(strategy),
        "speed": speed_ratio(strategy),
        "blast_radius": blast_radius_ratio(strategy),
    }
    total = sum(parts[axis] * weights[axis] for axis in weights)
    return round(total, 3), {axis: round(value, 3) for axis, value in parts.items()}


ranked = sorted(
    ((name, score(strategy)) for name, strategy in STRATEGIES.items()),
    key=lambda row: row[1][0],
    reverse=True,
)

for position, (name, (total, parts)) in enumerate(ranked, start=1):
    breakdown = "  ".join(f"{axis}={value:.2f}" for axis, value in sorted(parts.items()))
    print(f"{position}. {name:24s} total={total:.3f}   {breakdown}")

## Step 3: Read the job graphs off the scaffold's plan files

A strategy is only as good as the plan that implements it, and the plan is where the strategy is actually decided. This cell parses the `- type:` entries out of every plan shipped in `zap/templates/zap-dast-integration-scaffold/`, separating job stages from the assertion blocks nested inside each stage's `tests:` key, and reports per plan which stages run, whether an active scan is present, and whether discovery is ordered before active scanning.

The stage ordering matters: a plan that reaches the active scanner before the spider finishes is not the same scan as one that runs them in order, even when both list the same stages. The stage/test split matters too — reading every `- type:` line as a stage would report an alert-count assertion as a scan stage.

In [ ]:
SCAFFOLD_REL = Path("templates/zap-dast-integration-scaffold")


def resolve_scaffold():
    """Locate the ZAP scaffold from the notebook cwd, the kit root, or zap/."""
    candidates = (
        Path.cwd(),
        Path.cwd() / "zap",
        Path.cwd().parent,
        Path.cwd().parent.parent,
    )
    for base in candidates:
        candidate = base / SCAFFOLD_REL
        if candidate.is_dir():
            return candidate
    return None


def declared_types(plan_path):
    """Split the `- type:` entries of a plan into job stages and test blocks.

    Both use the same list syntax, but job stages are indented two spaces under
    `jobs:` while the assertions inside each stage's `tests:` block are indented
    deeper. Reporting them as one list would count an assertion as a scan stage.
    """
    stages, tests = [], []
    for line in plan_path.read_text(encoding="utf-8").splitlines():
        if not line.lstrip().startswith("- type:"):
            continue
        indent = len(line) - len(line.lstrip())
        value = line.strip().split(":", 1)[1].strip()
        (stages if indent <= 4 else tests).append(value)
    return stages, tests


def describe_order(stages):
    if "activeScan" not in stages or "spider" not in stages:
        return "no crawl/active pair to order"
    if stages.index("spider") < stages.index("activeScan"):
        return "discovery before active scan"
    return "active scan before discovery finishes"


scaffold = resolve_scaffold()
if scaffold is None:
    print("scaffold not resolved - run this notebook from the kit root or from zap/")
else:
    print(f"scaffold: {scaffold}\n")
    plans = sorted(scaffold.glob("*.yaml")) + sorted((scaffold / "plans").glob("*.yaml"))
    for plan in plans:
        stages, tests = declared_types(plan)
        print(plan.name)
        print(f"  stages        : {', '.join(stages) if stages else '(none found)'}")
        print(f"  tests         : {', '.join(tests) if tests else '(none found)'}")
        print(f"  active scan   : {'yes' if 'activeScan' in stages else 'no'}")
        print(f"  ordering      : {describe_order(stages)}")
        print()

## Step 4: Diff two reports instead of comparing counts

Comparing alert totals between two runs hides the only thing worth knowing: which findings appeared, which disappeared, and which were present in both. A promotion gate built on raw counts fails in both directions — it blocks on a run that added the same finding it already had, and it passes on a run that dropped a real finding because the crawler reached fewer URLs.

`compare_runs` works on the `alerts` list of a ZAP report and keys findings by alert reference plus risk. Pass real report paths to use it; the cell below runs it against clearly-labelled fixtures so the notebook stays executable without a scan.

In [ ]:
def load_report(path):
    with open(path, encoding="utf-8") as handle:
        return json.load(handle)


def alerts_by_risk(report):
    counts = {}
    for alert in report.get("alerts", []):
        risk = alert.get("risk")
        counts[risk] = counts.get(risk, 0) + 1
    return counts


def alert_key(alert):
    return (alert.get("alertRef"), alert.get("risk"))


def compare_runs(left, right, left_label, right_label):
    left_keys = {alert_key(alert) for alert in left.get("alerts", [])}
    right_keys = {alert_key(alert) for alert in right.get("alerts", [])}

    print(f"{left_label}: {len(left_keys)} distinct findings {alerts_by_risk(left)}")
    print(f"{right_label}: {len(right_keys)} distinct findings {alerts_by_risk(right)}")

    for title, keys in (
        ("added in " + right_label, sorted(right_keys - left_keys)),
        ("no longer reported in " + right_label, sorted(left_keys - right_keys)),
        ("present in both runs", sorted(left_keys & right_keys)),
    ):
        print(f"\n{title} ({len(keys)})")
        for ref, risk in keys:
            print(f"  {risk:<7} {ref}")

    return {
        "added": right_keys - left_keys,
        "dropped": left_keys - right_keys,
        "shared": left_keys & right_keys,
    }


# Fixtures standing in for two report files. The refs are placeholder labels, not
# scan output: replace with report paths produced by the report jobs in the plans.
BASELINE_REPORT = {
    "alerts": [
        {"alertRef": "fixture-header-strict-transport", "risk": "Medium"},
        {"alertRef": "fixture-cookie-attribute", "risk": "Low"},
        {"alertRef": "fixture-server-banner", "risk": "Low"},
    ]
}
ACTIVE_REPORT = {
    "alerts": [
        {"alertRef": "fixture-header-strict-transport", "risk": "Medium"},
        {"alertRef": "fixture-cookie-attribute", "risk": "Low"},
        {"alertRef": "fixture-input-handling", "risk": "High"},
        {"alertRef": "fixture-input-handling", "risk": "High"},
    ]
}

delta = compare_runs(
    BASELINE_REPORT,
    ACTIVE_REPORT,
    "passive-baseline report",
    "active-full report",
)

print("\ngate candidates (present in both runs):")
for ref, risk in sorted(delta["shared"]):
    print(f"  {risk:<7} {ref}")

## Step 5: Map strategies to environments

The scoring in Step 2 ranks the strategies abstractly; this cell binds the choice to a deployment context. Each entry records why the strategy fits — the ownership of the target, whether state changes are acceptable, and what the gate should assert. Read it as a decision procedure rather than a policy: the mapping is a starting point, and the entries a team overrides are the ones to argue about.

In [ ]:
ENVIRONMENTS = {
    "every staging deploy": {
        "ownership": "internal",
        "state_changes_acceptable": True,
        "strategy": "passive-baseline",
        "gate": "no new Medium or High findings versus the previous run",
        "why": "cheap enough to run on every deploy, and it catches the header and cookie regressions that reach production most often",
    },
    "release candidate": {
        "ownership": "internal",
        "state_changes_acceptable": True,
        "strategy": "active-full",
        "gate": "no unresolved High findings; the report is attached to the release record",
        "why": "the window where spending the scan budget on input-handling coverage pays off",
    },
    "documented api surface": {
        "ownership": "internal",
        "state_changes_acceptable": True,
        "strategy": "narrowed-context-active",
        "gate": "no unresolved High findings on the seeded endpoints",
        "why": "crawling an api surface wastes the budget rediscovering endpoints the inventory already lists",
    },
    "third-party or shared instance": {
        "ownership": "external",
        "state_changes_acceptable": False,
        "strategy": "passive-baseline",
        "gate": "advisory only; findings are triaged by hand",
        "why": "an active scan sends payloads at data the team does not own",
    },
}

for environment, spec in ENVIRONMENTS.items():
    print(f"{environment}")
    print(f"  strategy : {spec['strategy']}")
    print(f"  gate     : {spec['gate']}")
    print(f"  rationale: {spec['why']}")
    print()

chosen = {spec["strategy"] for spec in ENVIRONMENTS.values()}
print("strategies in active use:", ", ".join(sorted(chosen)))
print("strategies never used    :", ", ".join(sorted(set(STRATEGIES) - chosen)) or "none")

## Verify

Run the notebook top to bottom with no arguments. It should complete with no exceptions and print, in order:

1. The `STRATEGIES` attribute dump from Step 1.
2. A ranked table with one line per strategy from Step 2 — three lines, sorted by total.
3. The resolved scaffold path, then one block per plan file listing stages, tests, active-scan presence and ordering. If it prints `scaffold not resolved`, the notebook was started from a directory the resolver does not accept; re-run it from the kit root or from `zap/`.
4. From Step 4, an added / no-longer-reported / present-in-both breakdown. With the fixtures that is one added finding, one dropped finding and two present in both runs — the two identical High entries in the active fixture collapse to one distinct finding, which is the de-duplication the key function performs.
5. From Step 5, four environment blocks and a `strategies never used` line.

Substituting real reports: replace the two fixture dicts with `load_report("path/to/report-a.json")` and `load_report("path/to/report-b.json")`. Reports must come from report jobs with the same report template and equivalent contexts; comparing reports generated with different templates produces a diff that reflects the template, not the target.


## Common Errors

- **Active scan against a target that owns its data.** The active scanner submits forms and can trigger real actions. Symptoms are duplicate records, sent mail, or consumed one-time tokens — not scanner errors. Symptom-free and still wrong: a scan whose only state changes were destructive ones that returned errors the scanner counted as failures.
- **A plan that reaches the active scanner before discovery finishes.** Step 3 reports the ordering; when it reads `active scan before discovery finishes`, the active scan is working from a partial URL list and its coverage is not comparable to a plan that orders the stages.
- **Comparing alert counts across runs instead of diffing findings.** Counts conflate "new problem" with "same problem reported twice" and with "crawler reached fewer URLs". Step 4 keys findings by alert reference and risk for this reason.
- **Trusting a clean report from a truncated crawl.** A spider job with a duration budget that expires early yields fewer URLs, a smaller active scan, and a report with fewer alerts — indistinguishable from a fixed target by alert count alone. Coverage claims need URL counts from the spider statistics, not alert totals.
- **Narrowed-context scans seeded from a stale endpoint list.** The strategy is only faster than crawling if the list is current; a stale list produces a fast, clean, meaningless report. Treat regeneration of the endpoint list as part of the strategy, not as a prerequisite someone else owns.
- **Notebook run from an unexpected working directory.** Step 3's resolver accepts the kit root, `zap/`, or the notebook directory; anywhere else prints a resolution message instead of the plan contents.


## References

In-repo material this notebook builds on:

- `zap/docs/passive-vs-active-scanning-zap.md` — the passive-versus-active distinction and the environment-by-environment table this notebook formalises.
- `zap/docs/zap-automation-plan-structure.md` — job graph and context structure of automation-framework plans.
- `zap/docs/zap-scan-configuration-migration-guide.md` — how plan configuration changes are moved between targets and runners.
- `zap/templates/zap-dast-integration-scaffold/zap-automation-plan.yaml` and `zap/templates/zap-dast-integration-scaffold/plans/quick-scan.yaml` — the plan files read in Step 3.
- `zap/templates/zap-dast-integration-scaffold/scripts/run-zap-dast.sh` — how a plan is executed with target and report directory substituted.
